In [ ]:
"""
Integración Hugging Face para el Servidor MCP de Anamnesis Médica
Incluye modelos especializados en biomedicina y procesamiento de texto médico
"""

import asyncio
import httpx
import torch
from typing import Dict, List, Optional, Any, Tuple
from dataclasses import dataclass
import logging
import os
from datetime import datetime

# Hugging Face transformers
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    pipeline, BertTokenizer, BertForSequenceClassification
)
from sentence_transformers import SentenceTransformer
import numpy as np

# MCP integration
from mcp.server.fastmcp import FastMCP, Context
from mcp.server.session import ServerSession
from pydantic import BaseModel

logger = logging.getLogger(__name__)


@dataclass
class MedicalClassificationConfig:
    """Configuración para modelos de clasificación médica"""
    model_name: str
    confidence_threshold: float
    max_length: int = 512
    device: str = "cpu"


@dataclass
class EmbeddingConfig:
    """Configuración para modelos de embeddings"""
    model_name: str
    dimension: int
    max_length: int = 512
    device: str = "cpu"


class HuggingFaceService:
    """Servicio principal para integraciones con Hugging Face"""
    
    def __init__(self, api_token: Optional[str] = None):
        self.api_token = api_token or os.getenv("HUGGINGFACE_API_TOKEN")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Configuraciones de modelos médicos especializados
        self.classification_configs = {
            "biobert": MedicalClassificationConfig(
                model_name="dmis-lab/biobert-base-cased-v1.1",
                confidence_threshold=0.7,
                device=self.device
            ),
            "clinicalbert": MedicalClassificationConfig(
                model_name="emilyalsentzer/Bio_ClinicalBERT",
                confidence_threshold=0.7,
                device=self.device
            ),
            "pubmedbert": MedicalClassificationConfig(
                model_name="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
                confidence_threshold=0.6,
                device=self.device
            ),
            "medical_ner": MedicalClassificationConfig(
                model_name="d4data/biomedical-ner-all",
                confidence_threshold=0.8,
                device=self.device
            )
        }
        
        self.embedding_configs = {
            "biobert_embeddings": EmbeddingConfig(
                model_name="dmis-lab/biobert-base-cased-v1.1",
                dimension=768,
                device=self.device
            ),
            "sentence_biobert": EmbeddingConfig(
                model_name="pritamdeka/S-BioBert-snli-multinli-stsb",
                dimension=768,
                device=self.device
            ),
            "clinical_embeddings": EmbeddingConfig(
                model_name="emilyalsentzer/Bio_ClinicalBERT",
                dimension=768,
                device=self.device
            )
        }
        
        # Cache para modelos cargados
        self._models_cache = {}
        self._tokenizers_cache = {}
        self._pipelines_cache = {}
        
        logger.info(f"HuggingFace Service inicializado en dispositivo: {self.device}")

    async def load_classification_model(self, model_key: str) -> Tuple[Any, Any]:
        """Carga modelo de clasificación médica"""
        
        if model_key in self._models_cache:
            return self._models_cache[model_key], self._tokenizers_cache[model_key]
        
        config = self.classification_configs.get(model_key)
        if not config:
            raise ValueError(f"Modelo {model_key} no configurado")
        
        logger.info(f"Cargando modelo de clasificación: {config.model_name}")
        
        try:
            tokenizer = AutoTokenizer.from_pretrained(config.model_name)
            model = AutoModelForSequenceClassification.from_pretrained(config.model_name)
            
            if self.device == "cuda":
                model = model.to(self.device)
            
            self._models_cache[model_key] = model
            self._tokenizers_cache[model_key] = tokenizer
            
            logger.info(f"Modelo {config.model_name} cargado exitosamente")
            return model, tokenizer
            
        except Exception as e:
            logger.error(f"Error cargando modelo {config.model_name}: {e}")
            raise

    async def load_embedding_model(self, model_key: str) -> SentenceTransformer:
        """Carga modelo de embeddings médicos"""
        
        if model_key in self._models_cache:
            return self._models_cache[model_key]
        
        config = self.embedding_configs.get(model_key)
        if not config:
            raise ValueError(f"Modelo de embeddings {model_key} no configurado")
        
        logger.info(f"Cargando modelo de embeddings: {config.model_name}")
        
        try:
            # Usar SentenceTransformers para embeddings más eficientes
            model = SentenceTransformer(config.model_name, device=self.device)
            self._models_cache[model_key] = model
            
            logger.info(f"Modelo de embeddings {config.model_name} cargado exitosamente")
            return model
            
        except Exception as e:
            logger.error(f"Error cargando modelo de embeddings {config.model_name}: {e}")
            # Fallback a transformers básico
            return await self._load_basic_embedding_model(config)

    async def _load_basic_embedding_model(self, config: EmbeddingConfig):
        """Fallback para embeddings usando transformers básico"""
        tokenizer = AutoTokenizer.from_pretrained(config.model_name)
        model = AutoModel.from_pretrained(config.model_name)
        
        if self.device == "cuda":
            model = model.to(self.device)
        
        return {"model": model, "tokenizer": tokenizer}

    async def classify_symptoms_biobert(self, symptoms_text: str) -> Dict[str, Any]:
        """Clasifica síntomas usando BioBERT"""
        
        model, tokenizer = await self.load_classification_model("biobert")
        
        # Preparar texto para BioBERT
        inputs = tokenizer(
            symptoms_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )
        
        if self.device == "cuda":
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Realizar predicción
        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        
        # Procesar resultados
        predicted_class = torch.argmax(predictions, dim=-1).item()
        confidence = torch.max(predictions).item()
        
        return {
            "predicted_class": predicted_class,
            "confidence": confidence,
            "model_used": "biobert",
            "raw_predictions": predictions.cpu().numpy().tolist()
        }

    async def extract_medical_entities(self, text: str) -> List[Dict[str, Any]]:
        """Extrae entidades médicas usando NER especializado"""
        
        if "medical_ner" not in self._pipelines_cache:
            self._pipelines_cache["medical_ner"] = pipeline(
                "ner",
                model="d4data/biomedical-ner-all",
                tokenizer="d4data/biomedical-ner-all",
                aggregation_strategy="simple",
                device=0 if self.device == "cuda" else -1
            )
        
        ner_pipeline = self._pipelines_cache["medical_ner"]
        
        try:
            entities = ner_pipeline(text)
            
            # Procesar y filtrar entidades
            filtered_entities = []
            for entity in entities:
                if entity['score'] > 0.7:  # Filtro de confianza
                    filtered_entities.append({
                        "text": entity['word'],
                        "label": entity['entity_group'],
                        "confidence": entity['score'],
                        "start": entity.get('start', 0),
                        "end": entity.get('end', 0)
                    })
            
            return filtered_entities
            
        except Exception as e:
            logger.error(f"Error en extracción NER: {e}")
            return []

    async def generate_medical_embeddings(self, text: str, model_key: str = "sentence_biobert") -> List[float]:
        """Genera embeddings médicos especializados"""
        
        model = await self.load_embedding_model(model_key)
        
        try:
            if isinstance(model, SentenceTransformer):
                # Usar SentenceTransformers
                embeddings = model.encode([text], convert_to_numpy=True)
                return embeddings[0].tolist()
            else:
                # Usar transformers básico (fallback)
                tokenizer = model["tokenizer"]
                bert_model = model["model"]
                
                inputs = tokenizer(
                    text,
                    return_tensors="pt",
                    truncation=True,
                    padding=True,
                    max_length=512
                )
                
                if self.device == "cuda":
                    inputs = {k: v.to(self.device) for k, v in inputs.items()}
                
                with torch.no_grad():
                    outputs = bert_model(**inputs)
                    # Usar el token [CLS] para representación de la oración
                    cls_embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                
                return cls_embedding[0].tolist()
                
        except Exception as e:
            logger.error(f"Error generando embeddings: {e}")
            return []

    async def classify_medical_condition_advanced(self, anamnesis_data: Dict[str, Any]) -> Dict[str, Any]:
        """Clasificación médica avanzada combinando múltiples modelos"""
        
        # Preparar texto de entrada
        symptoms_text = self._prepare_medical_text(anamnesis_data)
        
        # 1. Extracción de entidades médicas
        medical_entities = await self.extract_medical_entities(symptoms_text)
        
        # 2. Clasificación con BioBERT
        biobert_results = await self.classify_symptoms_biobert(symptoms_text)
        
        # 3. Análisis de sentimientos médico (para urgencia/severidad)
        severity_analysis = await self._analyze_symptom_severity(symptoms_text)
        
        # 4. Búsqueda de condiciones similares usando embeddings
        similar_conditions = await self._find_similar_conditions(symptoms_text)
        
        # Combinar resultados
        combined_results = {
            "primary_classification": {
                "model": "biobert_ensemble",
                "predicted_conditions": self._map_biobert_to_conditions(biobert_results),
                "confidence": biobert_results["confidence"],
                "entities_found": len(medical_entities)
            },
            "medical_entities": medical_entities,
            "severity_indicators": severity_analysis,
            "similar_cases": similar_conditions,
            "recommendation_urgency": self._calculate_urgency_score(
                biobert_results, severity_analysis, medical_entities
            ),
            "model_info": {
                "primary_model": "BioBERT + Medical NER",
                "embedding_model": "Bio_ClinicalBERT",
                "processing_time": datetime.now().isoformat(),
                "device_used": self.device
            }
        }
        
        return combined_results

    def _prepare_medical_text(self, anamnesis_data: Dict[str, Any]) -> str:
        """Prepara texto médico para procesamiento"""
        
        components = []
        
        if chief_complaint := anamnesis_data.get("chief_complaint"):
            components.append(f"Chief complaint: {chief_complaint}")
        
        if symptoms := anamnesis_data.get("associated_symptoms"):
            components.append(f"Symptoms: {', '.join(symptoms)}")
        
        if duration := anamnesis_data.get("symptom_duration"):
            components.append(f"Duration: {duration}")
        
        if severity := anamnesis_data.get("symptom_severity"):
            components.append(f"Severity: {severity}/10")
        
        if history := anamnesis_data.get("medical_history"):
            components.append(f"Medical history: {', '.join(history)}")
        
        return " | ".join(components)

    async def _analyze_symptom_severity(self, text: str) -> Dict[str, Any]:
        """Analiza severidad de síntomas usando modelos de sentimiento médico"""
        
        # Palabras clave para severidad
        high_severity_keywords = [
            "severe", "intense", "unbearable", "excruciating", "emergency",
            "severo", "intenso", "insoportable", "emergencia"
        ]
        
        medium_severity_keywords = [
            "moderate", "persistent", "concerning", "worsening",
            "moderado", "persistente", "preocupante", "empeorando"
        ]
        
        text_lower = text.lower()
        
        severity_score = 0
        indicators = []
        
        for keyword in high_severity_keywords:
            if keyword in text_lower:
                severity_score += 3
                indicators.append(f"high_severity: {keyword}")
        
        for keyword in medium_severity_keywords:
            if keyword in text_lower:
                severity_score += 1
                indicators.append(f"medium_severity: {keyword}")
        
        return {
            "severity_score": min(severity_score, 10),
            "severity_level": "high" if severity_score >= 6 else "medium" if severity_score >= 3 else "low",
            "indicators": indicators
        }

    async def _find_similar_conditions(self, symptoms_text: str) -> List[Dict[str, Any]]:
        """Encuentra condiciones similares usando embeddings médicos"""
        
        # Generar embedding del caso actual
        case_embedding = await self.generate_medical_embeddings(symptoms_text)
        
        if not case_embedding:
            return []
        
        # Base de conocimiento médico simplificada (en producción, usar base de datos vectorial)
        medical_knowledge = [
            {
                "condition": "Infección respiratoria grave ",
                "typical_symptoms": "tos, dolor en garganta, congestion nasal, fatiga",
                "icd10": "J06.9"
            },
            {
                "condition": "bronquitis",
                "typical_symptoms": "tos persistente, incomodidiad en el pecho , produccion de moco ",
                "icd10": "J40"
            },
            {
                "condition": "Pneumonia",
                "typical_symptoms": "fever, cough, shortness of breath, chest pain",
                "icd10": "J18.9"
            }
        ]
        
        similarities = []
        for condition in medical_knowledge:
            condition_embedding = await self.generate_medical_embeddings(
                condition["typical_symptoms"]
            )
            
            if condition_embedding:
                # Calcular similaridad coseno
                similarity = self._cosine_similarity(case_embedding, condition_embedding)
                similarities.append({
                    "condition": condition["condition"],
                    "similarity": similarity,
                    "icd10": condition["icd10"],
                    "typical_symptoms": condition["typical_symptoms"]
                })
        
        # Ordenar por similaridad
        similarities.sort(key=lambda x: x["similarity"], reverse=True)
        return similarities[:5]  # Top 5 condiciones similares

    def _cosine_similarity(self, vec1: List[float], vec2: List[float]) -> float:
        """Calcula similaridad coseno entre dos vectores"""
        vec1_np = np.array(vec1)
        vec2_np = np.array(vec2)
        
        dot_product = np.dot(vec1_np, vec2_np)
        norm1 = np.linalg.norm(vec1_np)
        norm2 = np.linalg.norm(vec2_np)
        
        if norm1 == 0 or norm2 == 0:
            return 0.0
        
        return dot_product / (norm1 * norm2)

    def _map_biobert_to_conditions(self, biobert_results: Dict[str, Any]) -> List[Dict[str, Any]]:
        """Mapea resultados de BioBERT a condiciones médicas"""
        
        # Mapeo simplificado (en producción, usar mapeo más complejo)
        condition_mapping = {
            0: "Respiratory condition",
            1: "Cardiovascular condition", 
            2: "Gastrointestinal condition",
            3: "Neurological condition",
            4: "Other medical condition"
        }
        
        predicted_class = biobert_results["predicted_class"]
        confidence = biobert_results["confidence"]
        
        return [{
            "condition": condition_mapping.get(predicted_class, "Unknown condition"),
            "confidence": confidence,
            "classification_method": "biobert"
        }]

    def _calculate_urgency_score(self, classification: Dict, severity: Dict, entities: List) -> str:
        """Calcula score de urgencia basado en múltiples factores"""
        
        urgency_score = 0
        
        # Factor de confianza del modelo
        if classification["confidence"] > 0.8:
            urgency_score += 2
        elif classification["confidence"] > 0.6:
            urgency_score += 1
        
        # Factor de severidad
        severity_level = severity.get("severity_level", "low")
        if severity_level == "high":
            urgency_score += 3
        elif severity_level == "medium":
            urgency_score += 1
        
        # Factor de entidades críticas
        critical_entities = ["DISEASE", "SYMPTOM", "MEDICATION"]
        critical_count = sum(1 for entity in entities if entity["label"] in critical_entities)
        urgency_score += min(critical_count, 2)
        
        # Determinar nivel de urgencia
        if urgency_score >= 6:
            return "HIGH - Seek immediate medical attention"
        elif urgency_score >= 3:
            return "MEDIUM - Schedule medical consultation soon"
        else:
            return "LOW - Monitor symptoms and consult if worsening"

    async def use_inference_api(self, model_name: str, inputs: str) -> Dict[str, Any]:
        """Usa la API de Inference de Hugging Face para modelos no cargados localmente"""
        
        if not self.api_token:
            raise ValueError("API token de Hugging Face requerido")
        
        api_url = f"https://api-inference.huggingface.co/models/{model_name}"
        headers = {"Authorization": f"Bearer {self.api_token}"}
        
        async with httpx.AsyncClient() as client:
            try:
                response = await client.post(
                    api_url,
                    headers=headers,
                    json={"inputs": inputs},
                    timeout=30.0
                )
                
                if response.status_code == 200:
                    return response.json()
                else:
                    logger.error(f"API Error: {response.status_code} - {response.text}")
                    return {"error": f"API call failed: {response.status_code}"}
                    
            except Exception as e:
                logger.error(f"Error calling Hugging Face API: {e}")
                return {"error": str(e)}


# Integración con FastMCP
huggingface_service = HuggingFaceService()

mcp_hf = FastMCP(
    name="Medical Anamnesis with Hugging Face",
    instructions="""
    Agente de anamnesis médica potenciado por modelos especializados de Hugging Face.
    Utiliza BioBERT, ClinicalBERT y modelos NER médicos para análisis avanzado.
    """,
)


@mcp_hf.tool()
async def classify_symptoms_with_biobert(
    symptoms_text: str,
    ctx: Context[ServerSession, None]
) -> Dict[str, Any]:
    """Clasifica síntomas usando BioBERT especializado en biomedicina"""
    
    await ctx.info("Ejecutando clasificación con BioBERT...")
    await ctx.report_progress(0.2, message="Cargando modelo BioBERT")
    
    try:
        results = await huggingface_service.classify_symptoms_biobert(symptoms_text)
        await ctx.report_progress(1.0, message="Clasificación completada")
        
        return {
            "input_text": symptoms_text,
            "classification_results": results,
            "model_info": {
                "model_type": "BioBERT",
                "specialized_for": "biomedical_text",
                "confidence_threshold": 0.7
            }
        }
        
    except Exception as e:
        await ctx.error(f"Error en clasificación BioBERT: {e}")
        return {"error": str(e)}


@mcp_hf.tool()
async def extract_medical_entities_hf(
    medical_text: str,
    ctx: Context[ServerSession, None]
) -> Dict[str, Any]:
    """Extrae entidades médicas usando NER especializado de Hugging Face"""
    
    await ctx.info("Extrayendo entidades médicas...")
    await ctx.report_progress(0.3, message="Procesando texto médico")
    
    try:
        entities = await huggingface_service.extract_medical_entities(medical_text)
        await ctx.report_progress(1.0, message="Extracción completada")
        
        return {
            "input_text": medical_text,
            "entities_found": len(entities),
            "medical_entities": entities,
            "entity_types": list(set(entity["label"] for entity in entities))
        }
        
    except Exception as e:
        await ctx.error(f"Error en extracción NER: {e}")
        return {"error": str(e)}


@mcp_hf.tool()
async def generate_medical_embeddings_hf(
    text: str,
    model_type: str = "sentence_biobert",
    ctx: Context[ServerSession, None]
) -> Dict[str, Any]:
    """Genera embeddings médicos usando modelos especializados"""
    
    await ctx.info(f"Generando embeddings médicos con {model_type}...")
    await ctx.report_progress(0.4, message="Procesando texto")
    
    try:
        embeddings = await huggingface_service.generate_medical_embeddings(text, model_type)
        await ctx.report_progress(1.0, message="Embeddings generados")
        
        return {
            "input_text": text,
            "embedding_dimension": len(embeddings),
            "model_used": model_type,
            "embeddings": embeddings[:10],  # Solo mostrar primeros 10 valores
            "full_embedding_length": len(embeddings)
        }
        
    except Exception as e:
        await ctx.error(f"Error generando embeddings: {e}")
        return {"error": str(e)}


@mcp_hf.tool()
async def comprehensive_medical_analysis(
    patient_id: str,
    anamnesis_data: Dict[str, Any],
    ctx: Context[ServerSession, None]
) -> Dict[str, Any]:
    """Análisis médico comprehensivo usando múltiples modelos de Hugging Face"""
    
    await ctx.info(f"Iniciando análisis comprehensivo para paciente {patient_id}")
    await ctx.report_progress(0.1, message="Preparando análisis multi-modelo")
    
    try:
        # Análisis avanzado con múltiples modelos
        results = await huggingface_service.classify_medical_condition_advanced(anamnesis_data)
        await ctx.report_progress(1.0, message="Análisis completado")
        
        return {
            "patient_id": patient_id,
            "analysis_timestamp": datetime.now().isoformat(),
            "comprehensive_results": results,
            "disclaimer": """
            IMPORTANTE: Este análisis utiliza modelos de IA especializados en biomedicina,
            pero NO constituye un diagnóstico médico. Los resultados deben ser interpretados
            únicamente por profesionales médicos calificados.
            """,
            "models_used": [
                "BioBERT (clasificación)",
                "Medical NER (entidades)",
                "Bio_ClinicalBERT (embeddings)",
                "Severity Analysis (personalizado)"
            ]
        }
        
    except Exception as e:
        await ctx.error(f"Error en análisis comprehensivo: {e}")
        return {"error": str(e)}


@mcp_hf.resource("hf://model-info/{model_name}")
async def get_model_information(model_name: str) -> str:
    """Obtiene información sobre modelos de Hugging Face disponibles"""
    
    model_info = {
        "biobert": {
            "full_name": "dmis-lab/biobert-base-cased-v1.1",
            "description": "BioBERT pre-entrenado en literatura biomédica",
            "use_cases": ["clasificación médica", "NER biomédico"],
            "performance": "State-of-the-art en tareas biomédicas"
        },
        "clinicalbert": {
            "full_name": "emilyalsentzer/Bio_ClinicalBERT",
            "description": "BERT especializado en notas clínicas",
            "use_cases": ["análisis de historiales clínicos", "extracción de información médica"],
            "performance": "Optimizado para texto clínico real"
        },
        "pubmedbert": {
            "full_name": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
            "description": "BERT entrenado en abstracts de PubMed",
            "use_cases": ["investigación médica", "análisis de literatura científica"],
            "performance": "Excelente para terminología médica"
        }
    }
    
    info = model_info.get(model_name, {"error": "Modelo no encontrado"})
    return json.dumps(info, indent=2, ensure_ascii=False)


if __name__ == "__main__":
    # Para desarrollo, cargar modelos básicos
    import sys
    
    if "--preload-models" in sys.argv:
        print("Pre-cargando modelos de Hugging Face...")
        
        async def preload():
            try:
                await huggingface_service.load_classification_model("biobert")
                await huggingface_service.load_embedding_model("sentence_biobert")
                print("Modelos cargados exitosamente")
            except Exception as e:
                print(f"Error cargando modelos: {e}")
        
        asyncio.run(preload())
    
    mcp_hf.run()

SyntaxError: parameter without a default follows parameter with a default (1027426240.py, line 587)